# 02 — Exploratory Data Analysis (EDA)

## Objetivo

Realizar un análisis exploratorio de SpaPhish para identificar patrones y características asociados a los correos legítimos y de phishing.

El análisis se centra en:
- Distribución de las características de los correos.
- Estructura y longitud de los mensajes.
- Presencia de URLs y archivos adjuntos.
- Variables relacionadas con técnicas de ingeniería social.
- Relaciones entre las características disponibles y la etiqueta.

## Consideraciones

- SpaPhish será la fuente principal de esta exploración.
- No se combinarán sus etiquetas con las de SpamSpa.
- No se modificarán los datos originales.
- No se mostrarán ejemplos de mensajes ni información sensible.
- Las variables derivadas de anotaciones o relacionadas con la clasificación se analizarán de forma descriptiva, sin asumir que estarán disponibles para el modelo final.

**Qué hacemos:** Importamos las librerías necesarias y configuramos las opciones de visualización y reproducibilidad.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

**Qué hacemos:** Localizamos y cargamos el CSV de SpaPhish desde la estructura del proyecto, sin modificar el archivo original.

In [ ]:
PROJECT_ROOT = Path.cwd()

if (PROJECT_ROOT / "ml" / "data" / "raw").exists():
    DATA_DIR = PROJECT_ROOT / "ml" / "data" / "raw"
elif (PROJECT_ROOT.parent / "data" / "raw").exists():
    DATA_DIR = PROJECT_ROOT.parent / "data" / "raw"
else:
    raise FileNotFoundError("Dataset directory not found.")

SPAPHISH_PATH = DATA_DIR / "spaphish" / "Spaphish dataset - DiB.csv"

if not SPAPHISH_PATH.exists():
    raise FileNotFoundError(f"SpaPhish not found: {SPAPHISH_PATH}")

spaphish = pd.read_csv(
    SPAPHISH_PATH,
    sep=",",
    encoding="utf-8",
    on_bad_lines="error"
)

print(f"Dataset loaded: {spaphish.shape[0]} rows, {spaphish.shape[1]} columns")

**Qué hacemos:** Revisamos las columnas, sus tipos de datos y los valores ausentes para conocer la estructura del dataset.

In [ ]:
column_overview = pd.DataFrame({
    "column": spaphish.columns,
    "dtype": spaphish.dtypes.astype(str).values,
    "missing_values": spaphish.isna().sum().values,
    "missing_percentage": (
        spaphish.isna().mean().values * 100
    ).round(2)
})

display(column_overview)

## 1. Análisis de las características generales

En esta sección se estudian las variables numéricas del dataset para conocer sus rangos, distribuciones y posibles diferencias entre los correos legítimos y los clasificados como phishing.

Se excluyen los identificadores y las columnas de texto libre para centrarnos en las características cuantificables.

**Qué hacemos:** Calculamos estadísticas descriptivas de las variables numéricas para conocer sus rangos y dispersión.

In [ ]:
numeric_columns = spaphish.select_dtypes(include="number").columns.tolist()

numeric_summary = spaphish[numeric_columns].describe().T

display(numeric_summary.round(2))


**Qué hacemos:** Comprobamos los valores distintos de cada variable numérica para identificar su escala y naturaleza.

In [ ]:
for column in numeric_columns:
    print(f"{column}: {sorted(spaphish[column].dropna().unique().tolist())}")


## 2. Características técnicas de los correos

Se analiza la presencia y frecuencia de elementos técnicos que pueden aportar información sobre el mensaje:

- Número de URLs.
- Número de archivos adjuntos.
- Tamaño total de los archivos adjuntos.
- Número de saltos o intermediarios registrados (`hops_count`).

Se comparan estas características entre correos legítimos y phishing para identificar posibles patrones.

**Qué hacemos:** Visualizamos las distribuciones de URLs, adjuntos, tamaño de adjuntos y hops, separadas por etiqueta.

In [ ]:
technical_features = [
    "url_count",
    "attachments_count",
    "attachments_total_size",
    "hops_count"
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for column, ax in zip(technical_features, axes.flatten()):
    sns.histplot(
        data=spaphish,
        x=column,
        hue="Label",
        bins=30,
        multiple="layer",
        element="step",
        stat="count",
        common_norm=False,
        ax=ax
    )
    ax.set_title(f"Distribución de {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Número de correos")

plt.tight_layout()
plt.show()


**Qué hacemos:** Comparamos media, mediana y desviación estándar de las características técnicas entre las dos clases.

In [ ]:
technical_by_label = (
    spaphish
    .groupby("Label")[technical_features]
    .agg(["mean", "median", "std"])
    .round(2)
)

display(technical_by_label)


**Qué hacemos:** Creamos indicadores de presencia de URLs y adjuntos y calculamos su porcentaje por clase.

In [ ]:
spaphish_eda = spaphish.copy()

spaphish_eda["has_urls"] = spaphish_eda["url_count"] > 0
spaphish_eda["has_attachments"] = spaphish_eda["attachments_count"] > 0

presence_features = ["has_urls", "has_attachments"]

presence_by_label = (
    spaphish_eda
    .groupby("Label")[presence_features]
    .mean()
    .mul(100)
    .round(2)
)

display(presence_by_label)

**Qué hacemos:** Representamos los porcentajes de correos con URLs y con archivos adjuntos para comparar ambas clases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for column, ax in zip(presence_features, axes):
    plot_data = (
        spaphish_eda
        .groupby("Label")[column]
        .mean()
        .mul(100)
        .reset_index()
    )

    sns.barplot(
        data=plot_data,
        x="Label",
        y=column,
        ax=ax
    )

    ax.set_title(
        "Correos con URLs" if column == "has_urls"
        else "Correos con archivos adjuntos"
    )
    ax.set_xlabel("Etiqueta (0 = legítimo, 1 = phishing)")
    ax.set_ylabel("Porcentaje de correos (%)")

plt.tight_layout()
plt.show()


### 2.1. Conclusiones preliminares: características técnicas

El análisis muestra diferencias entre los correos legítimos y los clasificados como phishing:

- La presencia de URLs es elevada en ambas categorías, pero especialmente en los correos de phishing.
- Los correos legítimos presentan una mayor frecuencia de archivos adjuntos.
- El tamaño total de los adjuntos muestra una distribución muy asimétrica, con una gran concentración de valores en cero y algunos valores extremos.
- La variable `hops_count` presenta distribuciones solapadas y diferencias moderadas entre las clases.

Estas observaciones son descriptivas y corresponden exclusivamente al dataset analizado. No permiten establecer relaciones causales ni garantizan que estas características generalicen a otros conjuntos de correos.

Será necesario evaluar su utilidad conjunta durante el entrenamiento del clasificador.

**Qué hacemos:** Comprobamos los valores posibles de las anotaciones A/B/C y de las variables agregadas de ingeniería social.

In [ ]:
annotation_features = [
    "authority",
    "social_proof",
    "liking_similarity_deception",
    "commitment_integrity_reciprocation",
    "distraction"
]

annotation_blocks = [
    f"{feature}_{block}"
    for feature in annotation_features
    for block in ["A", "B", "C"]
]

for column in annotation_blocks + annotation_features:
    print(f"{column}: {sorted(spaphish[column].dropna().unique().tolist())}")

## 4. Análisis de las técnicas de ingeniería social

SpaPhish incluye cinco dimensiones relacionadas con técnicas de persuasión e ingeniería social:

- Authority: autoridad.
- Social proof: prueba social.
- Liking, similarity and deception: simpatía, similitud y engaño.
- Commitment, integrity and reciprocation: compromiso, integridad y reciprocidad.
- Distraction: distracción.

Cada dimensión cuenta con tres anotaciones (A, B y C), además de una variable agregada.

Se analizará la frecuencia de estas características en cada clase y se comprobará la relación entre las anotaciones individuales y las variables agregadas.

**Qué hacemos:** Calculamos el porcentaje de presencia de cada técnica de ingeniería social por clase usando las variables agregadas.

In [ ]:
annotation_labels = {
    "authority": "Autoridad",
    "social_proof": "Prueba social",
    "liking_similarity_deception": "Simpatía, similitud y engaño",
    "commitment_integrity_reciprocation": "Compromiso, integridad y reciprocidad",
    "distraction": "Distracción"
}

annotation_prevalence = (
    spaphish
    .groupby("Label")[annotation_features]
    .mean()
    .mul(100)
    .round(2)
)

annotation_prevalence = annotation_prevalence.rename(
    columns=annotation_labels
)

display(annotation_prevalence.T)


**Qué hacemos:** Representamos la frecuencia de las cinco técnicas de ingeniería social y comparamos legítimos y phishing.

In [ ]:
plot_data = (
    spaphish
    .groupby("Label")[annotation_features]
    .mean()
    .mul(100)
    .rename(columns=annotation_labels)
    .reset_index()
    .melt(
        id_vars="Label",
        var_name="Técnica",
        value_name="Porcentaje"
    )
)

plt.figure(figsize=(13, 6))

sns.barplot(
    data=plot_data,
    x="Técnica",
    y="Porcentaje",
    hue="Label"
)

plt.title("Presencia de técnicas de ingeniería social por clase")
plt.xlabel("Técnica")
plt.ylabel("Correos con la técnica (%)")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Etiqueta", labels=["Legítimo (0)", "Phishing (1)"])
plt.tight_layout()
plt.show()


**Qué hacemos:** Calculamos la frecuencia de cada técnica en los bloques de anotación A, B y C, separada por clase.

In [ ]:
block_records = []

for feature in annotation_features:
    for block in ["A", "B", "C"]:
        column = f"{feature}_{block}"

        for label, group in spaphish.groupby("Label"):
            block_records.append({
                "feature": annotation_labels[feature],
                "block": block,
                "Label": label,
                "percentage": group[column].mean() * 100
            })

block_prevalence = pd.DataFrame(block_records)

display(
    block_prevalence
    .pivot_table(
        index=["feature", "block"],
        columns="Label",
        values="percentage"
    )
    .round(2)
)


**Qué hacemos:** Comparamos visualmente las anotaciones A/B/C de cada técnica entre correos legítimos y phishing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, label in zip(axes, [0, 1]):
    subset = block_prevalence[
        block_prevalence["Label"] == label
    ]

    sns.barplot(
        data=subset,
        x="feature",
        y="percentage",
        hue="block",
        ax=ax
    )

    ax.set_title(
        "Correos legítimos" if label == 0 else "Correos de phishing"
    )
    ax.set_xlabel("Técnica")
    ax.set_ylabel("Presencia (%)")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

Comprobamos si cada variable agregada coincide con la presencia de la técnica en al menos uno de los tres bloques. Esta es una comprobación de consistencia, no una afirmación sobre el método utilizado por los autores para crear las columnas agregadas.

**Qué hacemos:** Medimos el grado de coincidencia entre cada variable agregada y la presencia de la técnica en al menos uno de los bloques A/B/C.

In [ ]:
consistency_records = []

for feature in annotation_features:
    block_columns = [f"{feature}_{block}" for block in ["A", "B", "C"]]

    block_any = spaphish[block_columns].max(axis=1)
    aggregate = spaphish[feature]

    consistency_records.append({
        "feature": annotation_labels[feature],
        "matching_records": (block_any == aggregate).sum(),
        "total_records": len(spaphish),
        "agreement_percentage": (block_any == aggregate).mean() * 100
    })

consistency_df = pd.DataFrame(consistency_records)

display(consistency_df.round(2))

**Qué hacemos:** Desglosamos las discrepancias para distinguir los casos en que coinciden ambas representaciones y aquellos en que difieren.

In [ ]:
discrepancy_records = []

for feature in annotation_features:
    block_columns = [f"{feature}_{block}" for block in ["A", "B", "C"]]

    block_any = spaphish[block_columns].max(axis=1)
    aggregate = spaphish[feature]

    discrepancy_records.append({
        "feature": annotation_labels[feature],
        "both_present": ((block_any == 1) & (aggregate == 1)).sum(),
        "both_absent": ((block_any == 0) & (aggregate == 0)).sum(),
        "blocks_present_aggregate_absent": (
            (block_any == 1) & (aggregate == 0)
        ).sum(),
        "blocks_absent_aggregate_present": (
            (block_any == 0) & (aggregate == 1)
        ).sum()
    })

discrepancy_df = pd.DataFrame(discrepancy_records)

display(discrepancy_df)

## 5. Análisis temporal de los correos

Se estudia la distribución temporal de los mensajes a partir de la columna `date`.

El objetivo es conocer el periodo que cubre el dataset y comprobar si existen diferencias en la distribución temporal de los correos legítimos y de phishing.

Hay 24 fechas ausentes identificadas durante la auditoría inicial. También se comprobará si existen valores que no puedan convertirse correctamente a formato fecha.

Este análisis es descriptivo: no pretende identificar tendencias generales del phishing ni establecer una evolución temporal de las amenazas.

Convertimos la columna date a formato fecha y cuantificamos los registros válidos, ausentes o no interpretables, sin mostrar fechas individuales.

**Qué hacemos:** Convertimos las fechas a formato datetime y comprobamos cuántas son válidas, ausentes o no interpretables.

In [ ]:
spaphish_eda["parsed_date"] = pd.to_datetime(
    spaphish_eda["date"],
    format="mixed",
    errors="coerce",
    utc=True
)

date_quality = pd.DataFrame({
    "metric": [
        "Total records",
        "Original missing dates",
        "Valid parsed dates",
        "Unparseable non-missing dates"
    ],
    "count": [
        len(spaphish_eda),
        spaphish_eda["date"].isna().sum(),
        spaphish_eda["parsed_date"].notna().sum(),
        (
            spaphish_eda["date"].notna()
            & spaphish_eda["parsed_date"].isna()
        ).sum()
    ]
})

display(date_quality)

valid_dates = spaphish_eda["parsed_date"].dropna()

if not valid_dates.empty:
    print("Date range:")
    print(valid_dates.min(), "to", valid_dates.max())


Agrupamos los correos por año y etiqueta para comprobar cómo se distribuyen ambas clases en el periodo disponible.

**Qué hacemos:** Contamos los correos por año y etiqueta y representamos la distribución anual de ambas clases.

In [ ]:
spaphish_eda["year"] = spaphish_eda["parsed_date"].dt.year

year_distribution = (
    spaphish_eda
    .dropna(subset=["year"])
    .groupby(["year", "Label"])
    .size()
    .unstack(fill_value=0)
)

display(year_distribution)

year_distribution.plot(
    kind="bar",
    figsize=(11, 5),
    width=0.8
)

plt.title("Distribución anual de correos por clase")
plt.xlabel("Año")
plt.ylabel("Número de correos")
plt.legend(
    title="Etiqueta",
    labels=["Legítimo (0)", "Phishing (1)"]
)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Analizamos el volumen mensual de registros para observar si hay concentraciones temporales que debamos tener en cuenta al interpretar el dataset.

**Qué hacemos:** Agrupamos los registros por mes y etiqueta para visualizar su distribución temporal.

In [ ]:
monthly_distribution = (
    spaphish_eda
    .dropna(subset=["parsed_date"])
    .assign(
        year_month=lambda df: df["parsed_date"].dt.strftime("%Y-%m")
    )
    .groupby(["year_month", "Label"])
    .size()
    .unstack(fill_value=0)
)

display(monthly_distribution)

monthly_distribution.plot(
    figsize=(14, 5),
    marker="o"
)

plt.title("Distribución mensual de correos por clase")
plt.xlabel("Año y mes")
plt.ylabel("Número de correos")
plt.legend(
    title="Etiqueta",
    labels=["Legítimo (0)", "Phishing (1)"]
)
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

### 5.1. Proporción de clases por año

Dado que el número de registros varía considerablemente entre años, calculamos la proporción de correos legítimos y phishing dentro de cada año.

Esta comparación permite observar cómo cambia la composición de la muestra sin confundir el volumen de registros con la proporción de cada clase.

Calculamos el porcentaje de cada clase dentro de cada año y lo representamos mediante un gráfico de barras apiladas.

**Qué hacemos:** Calculamos la proporción de cada clase dentro de cada año y la representamos con barras apiladas.

In [ ]:
year_counts = (
    spaphish_eda
    .dropna(subset=["year"])
    .groupby(["year", "Label"])
    .size()
    .unstack(fill_value=0)
)

year_percentages = (
    year_counts
    .div(year_counts.sum(axis=1), axis=0)
    .mul(100)
)

display(year_percentages.round(2))

year_percentages.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 5)
)

plt.title("Proporción de clases por año")
plt.xlabel("Año")
plt.ylabel("Porcentaje de correos (%)")
plt.legend(
    title="Etiqueta",
    labels=["Legítimo (0)", "Phishing (1)"]
)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 6. Análisis de las características textuales

En esta sección se estudian las características del asunto y del cuerpo de los correos electrónicos.

El objetivo es identificar diferencias estructurales entre los mensajes legítimos y los de phishing, prestando atención a:

- Longitud del asunto.
- Longitud del cuerpo.
- Longitud total del mensaje.
- Distribución de las longitudes según la clase.
- Presencia de asuntos vacíos.

No se mostrarán ejemplos de mensajes originales para evitar exponer información potencialmente sensible.

Creamos una copia de trabajo y calculamos el número de caracteres del asunto, el cuerpo y el mensaje completo. Los valores ausentes del asunto se contabilizan como longitud cero, sin modificar el dataset original.

**Qué hacemos:** Calculamos la longitud en caracteres del asunto, del cuerpo y del texto completo y resumimos sus distribuciones por clase.

In [ ]:
text_eda = spaphish_eda.copy()

text_eda["subject_length"] = (
    text_eda["subject"].fillna("").str.len()
)

text_eda["body_length"] = (
    text_eda["body"].fillna("").str.len()
)

text_eda["total_text_length"] = (
    text_eda["subject"].fillna("").str.len()
    + text_eda["body"].fillna("").str.len()
)

text_features = [
    "subject_length",
    "body_length",
    "total_text_length"
]

print("Empty subjects:", (text_eda["subject_length"] == 0).sum())
print("Empty bodies:", (text_eda["body_length"] == 0).sum())

display(
    text_eda
    .groupby("Label")[text_features]
    .agg(["mean", "median", "std", "min", "max"])
    .round(2)
)

Representamos las distribuciones mediante diagramas de caja para comparar las longitudes y detectar valores extremos. Utilizamos escala logarítmica en el eje vertical para facilitar la lectura de distribuciones muy asimétricas.

**Qué hacemos:** Comparamos las longitudes del asunto, cuerpo y mensaje completo mediante diagramas de caja.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for column, ax in zip(text_features, axes):
    sns.boxplot(
        data=text_eda,
        x="Label",
        y=column,
        ax=ax
    )

    ax.set_title(column.replace("_", " ").title())
    ax.set_xlabel("Etiqueta (0 = legítimo, 1 = phishing)")
    ax.set_ylabel("Número de caracteres")
    ax.set_yscale("symlog", linthresh=10)

plt.suptitle("Longitud de los correos según su clasificación")
plt.tight_layout()
plt.show()


Comparamos las distribuciones de longitud mediante histogramas para observar dónde se concentra la mayor parte de los mensajes.

**Qué hacemos:** Representamos las distribuciones de longitud con histogramas normalizados por clase.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for column, ax in zip(text_features, axes):
    sns.histplot(
        data=text_eda,
        x=column,
        hue="Label",
        bins=40,
        element="step",
        stat="density",
        common_norm=False,
        ax=ax
    )

    ax.set_title(column.replace("_", " ").title())
    ax.set_xlabel("Número de caracteres")
    ax.set_ylabel("Densidad")

plt.tight_layout()
plt.show()


### Conclusiones preliminares: análisis temporal y textual

**Análisis temporal**

- El 98,28 % de los registros tiene una fecha válida.
- El dataset cubre el periodo comprendido entre julio de 2014 y diciembre de 2025.
- La proporción de correos de phishing aumenta notablemente en los últimos años de la muestra.
- Esta distribución puede estar relacionada con el proceso de recopilación de los datos y no debe interpretarse como una tendencia real del phishing.
- La fecha no se incorporará directamente como predictor del primer baseline sin una justificación adicional.

**Análisis textual**

- Los asuntos de phishing presentan una longitud media y mediana superiores a los legítimos.
- Los cuerpos de los correos legítimos son considerablemente más largos.
- La longitud total también es superior en los correos legítimos.
- Se han identificado tres asuntos vacíos y ningún cuerpo vacío.
- Las distribuciones presentan asimetría y valores extremos, por lo que la mediana resulta especialmente útil para describirlas.

**Implicaciones para el modelo**

El asunto y el cuerpo deben analizarse como componentes diferenciados. Las diferencias observadas justifican explorar características lingüísticas adicionales, aunque ninguna de estas variables debe considerarse suficiente por sí sola para identificar phishing.

## 7. Análisis de características lingüísticas

Se estudian características superficiales del lenguaje para identificar posibles diferencias de estructura y estilo entre los correos legítimos y los de phishing.

Se calculan indicadores como el número de palabras, el uso de mayúsculas, la presencia de cifras y determinados signos de puntuación.

El análisis se realiza de forma agregada, sin mostrar ni almacenar ejemplos de mensajes originales en las salidas del notebook.

Estas características son exploratorias y no representan por sí solas indicadores concluyentes de fraude.

Creamos variables numéricas a partir del asunto y del cuerpo. Las funciones trabajan sobre copias del texto y devuelven únicamente recuentos, nunca el contenido original.

**Qué hacemos:** Extraemos recuentos y proporciones lingüísticas del asunto y del cuerpo sin mostrar el texto original.

In [ ]:
import re
import string

def count_words(text):
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))


def count_uppercase(text):
    return sum(char.isupper() for char in text)


def count_letters(text):
    return sum(char.isalpha() for char in text)


def count_digits(text):
    return sum(char.isdigit() for char in text)


def extract_text_features(df, source_column, prefix):
    texts = df[source_column].fillna("").astype(str)

    result = pd.DataFrame(index=df.index)

    result[f"{prefix}_word_count"] = texts.apply(count_words)
    result[f"{prefix}_uppercase_count"] = texts.apply(count_uppercase)
    result[f"{prefix}_letter_count"] = texts.apply(count_letters)
    result[f"{prefix}_digit_count"] = texts.apply(count_digits)

    result[f"{prefix}_uppercase_ratio"] = (
        result[f"{prefix}_uppercase_count"]
        / result[f"{prefix}_letter_count"].replace(0, np.nan)
    ).fillna(0)

    result[f"{prefix}_exclamation_count"] = texts.str.count("!")
    result[f"{prefix}_question_count"] = texts.str.count(r"\?")
    result[f"{prefix}_punctuation_count"] = texts.apply(
        lambda text: sum(char in string.punctuation for char in text)
    )

    return result


subject_features = extract_text_features(
    text_eda, "subject", "subject"
)

body_features = extract_text_features(
    text_eda, "body", "body"
)

linguistic_features = pd.concat(
    [text_eda[["Label"]], subject_features, body_features],
    axis=1
)

print("Extracted linguistic features:", linguistic_features.shape[1] - 1)
print("Original message content displayed: No")

Comparamos medias y medianas de las características lingüísticas para identificar diferencias entre mensajes legítimos y phishing.

**Qué hacemos:** Calculamos medias y medianas de las características lingüísticas para compararlas entre clases.

In [ ]:
linguistic_columns = [
    column for column in linguistic_features.columns
    if column != "Label"
]

linguistic_summary = (
    linguistic_features
    .groupby("Label")[linguistic_columns]
    .agg(["mean", "median"])
    .T
)

display(linguistic_summary.round(3))


Representamos las diferencias entre clases mediante diagramas de caja. Limitamos los gráficos a variables seleccionadas para facilitar su interpretación.

**Qué hacemos:** Visualizamos la distribución de las características lingüísticas seleccionadas mediante diagramas de caja.

In [ ]:
features_to_plot = [
    "subject_word_count",
    "body_word_count",
    "subject_uppercase_ratio",
    "body_uppercase_ratio",
    "subject_digit_count",
    "body_digit_count",
    "subject_exclamation_count",
    "body_exclamation_count",
    "subject_question_count",
    "body_question_count"
]

fig, axes = plt.subplots(2, 5, figsize=(18, 9))

for column, ax in zip(features_to_plot, axes.flatten()):
    sns.boxplot(
        data=linguistic_features,
        x="Label",
        y=column,
        ax=ax
    )

    ax.set_title(column.replace("_", " ").title(), fontsize=9)
    ax.set_xlabel("Clase (0 = legítimo, 1 = phishing)")
    ax.set_ylabel("")

plt.suptitle("Características lingüísticas por clase")
plt.tight_layout()
plt.show()


### 7.1. Características lingüísticas normalizadas

Los recuentos absolutos pueden estar condicionados por la longitud de cada mensaje. Para realizar comparaciones más justas, calculamos la frecuencia de palabras, dígitos y signos de puntuación por cada 100 caracteres.

De esta forma podemos distinguir si una clase utiliza más determinados elementos proporcionalmente, independientemente de la longitud del texto.

Calculamos tasas normalizadas para el asunto y el cuerpo. Así podremos comparar el uso relativo de números y signos de puntuación entre clases.

**Qué hacemos:** Normalizamos los recuentos de palabras, dígitos y signos por cada 100 caracteres para controlar la longitud del texto.

In [ ]:
normalized_features = linguistic_features[["Label"]].copy()

for prefix in ["subject", "body"]:
    length = text_eda[f"{prefix}_length"].replace(0, np.nan)

    normalized_features[f"{prefix}_words_per_100_chars"] = (
        linguistic_features[f"{prefix}_word_count"] / length * 100
    )

    for feature in [
        "digit_count",
        "exclamation_count",
        "question_count",
        "punctuation_count"
    ]:
        normalized_features[f"{prefix}_{feature}_per_100_chars"] = (
            linguistic_features[f"{prefix}_{feature}"] / length * 100
        )

normalized_features = normalized_features.replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

normalized_columns = [
    column for column in normalized_features.columns
    if column != "Label"
]

normalized_summary = (
    normalized_features
    .groupby("Label")[normalized_columns]
    .agg(["mean", "median"])
    .T
)

display(normalized_summary.round(3))

Comparamos mediante diagramas de caja las tasas normalizadas. Esto ayuda a comprobar si las diferencias observadas anteriormente se mantienen al controlar la longitud del mensaje.

**Qué hacemos:** Comparamos visualmente las frecuencias lingüísticas normalizadas entre correos legítimos y phishing.

In [ ]:
features_to_plot = [
    "subject_words_per_100_chars",
    "subject_digit_count_per_100_chars",
    "subject_exclamation_count_per_100_chars",
    "subject_punctuation_count_per_100_chars",
    "body_words_per_100_chars",
    "body_digit_count_per_100_chars",
    "body_exclamation_count_per_100_chars",
    "body_question_count_per_100_chars",
    "body_punctuation_count_per_100_chars"
]

fig, axes = plt.subplots(2, 5, figsize=(18, 9))

for column, ax in zip(features_to_plot, axes.flatten()):
    sns.boxplot(
        data=normalized_features,
        x="Label",
        y=column,
        ax=ax
    )

    ax.set_title(column.replace("_", " ").title(), fontsize=8)
    ax.set_xlabel("Clase (0 = legítimo, 1 = phishing)")
    ax.set_ylabel("Frecuencia por 100 caracteres")

axes.flatten()[-1].set_visible(False)

plt.suptitle("Características lingüísticas normalizadas")
plt.tight_layout()
plt.show()

### 7.2. Relaciones entre características lingüísticas

Se analiza la correlación entre las características extraídas para identificar variables que puedan aportar información similar o presentar relaciones entre sí.

Este análisis permite detectar posibles redundancias antes de seleccionar las variables que se utilizarán en los experimentos de clasificación.

Calculamos las correlaciones de Pearson entre las características lingüísticas y las representamos mediante un mapa de calor. Excluimos la etiqueta para centrarnos en las relaciones entre predictores.

**Qué hacemos:** Calculamos y visualizamos las correlaciones entre las características lingüísticas para detectar posibles redundancias.

In [ ]:
correlation_matrix = linguistic_features.drop(
    columns="Label"
).corr()

plt.figure(figsize=(14, 11))

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0,
    annot=False,
    linewidths=0.3
)

plt.title("Correlación entre características lingüísticas")
plt.tight_layout()
plt.show()


## 8. Conclusiones generales de la EDA

## 8.1. Características técnicas

- La presencia de URLs es elevada en ambas clases y especialmente frecuente en phishing.
- Los correos legítimos presentan una mayor presencia de archivos adjuntos.
- El tamaño de los adjuntos presenta una distribución muy asimétrica.
- `hops_count` muestra diferencias moderadas y bastante solapamiento entre clases.

## 8.2. Técnicas de ingeniería social

- La distracción presenta una diferencia descriptiva especialmente marcada entre legítimos y phishing.
- Autoridad y simpatía, similitud y engaño aparecen con frecuencia en ambas clases.
- Prueba social y compromiso, integridad y reciprocidad son más frecuentes en los correos legítimos de esta muestra.
- Las variables agregadas no coinciden completamente con la unión de las anotaciones A/B/C. Sus definiciones deben revisarse antes de utilizarlas.

## 8.3. Características temporales

- El 98,28 % de los registros tiene fechas válidas.
- La distribución de clases cambia notablemente a lo largo del periodo estudiado.
- En 2024 y 2025 predominan los registros de phishing.
- La fecha puede introducir sesgos relacionados con la recopilación del dataset y no se utilizará directamente como predictor del primer baseline.

## 8.4. Características textuales y lingüísticas

- Los asuntos de phishing son más largos en términos de caracteres y palabras.
- Los cuerpos legítimos son considerablemente más extensos.
- La densidad de palabras por cada 100 caracteres es similar entre ambas clases.
- Los asuntos de phishing presentan una mayor frecuencia relativa de dígitos, exclamaciones y puntuación.
- Algunas características numéricas están correlacionadas entre sí, especialmente las relacionadas con la longitud del cuerpo.

## 8.5. Limitaciones

- Los resultados describen únicamente los registros de SpaPhish y no garantizan que los patrones se mantengan en otros conjuntos de datos.
- La distribución temporal y las diferencias entre clases pueden estar condicionadas por el proceso de recopilación.
- Las anotaciones de ingeniería social requieren conocer su disponibilidad y método de obtención antes de utilizarlas como predictores.
- Las características exploradas no permiten establecer relaciones causales.
- Las distribuciones presentan asimetrías y valores extremos que deben considerarse en los análisis posteriores.

# 9. Recomendaciones para el desarrollo del clasificador

## Preparación de datos

1. Mantener separados el asunto y el cuerpo durante la preparación del texto.
2. Revisar los duplicados y posibles coincidencias entre los conjuntos de entrenamiento y evaluación.
3. Conservar los registros con asuntos ausentes mediante un tratamiento coherente y documentado.
4. Revisar las variables con valores extremos y distribuciones muy asimétricas.
5. Documentar el tratamiento de las fechas y no utilizarlas automáticamente como variables predictoras.
6. Mantener trazabilidad de las transformaciones y de la distribución de clases.

## Clasificador baseline

1. Utilizar inicialmente TF-IDF sobre el contenido textual.
2. Comparar el uso del asunto y el cuerpo por separado y conjuntamente.
3. Evaluar Logistic Regression, SVM y Random Forest.
4. Analizar precision, recall y F1 por clase, prestando especial atención al recall de phishing.
5. Evaluar las características numéricas como un experimento adicional, no como una fuente de información garantizada.
6. Evitar utilizar anotaciones manuales que no estén disponibles para un correo nuevo.
7. Mantener una evaluación reproducible y controlar los posibles efectos de la distribución temporal.

## Consideración final

La EDA identifica patrones útiles para orientar los experimentos, pero la utilidad real de cada característica deberá comprobarse mediante validación y evaluación del clasificador.